In [2]:
from huggingface_hub import login, hf_hub_download, HfApi
from huggingface_hub.utils import RepositoryNotFoundError
import os

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN   = secrets.get_secret("HF_TOKEN")   # store token in Kaggle Secrets, not plaintext
HF_REPO_ID = "space-cadet/wavlm-ser"
SAVE_FOLDER = "/kaggle/working/emotion_classification/results/ser_wavlm/1986/save"

login(token=HF_TOKEN)
os.makedirs(SAVE_FOLDER, exist_ok=True)

try:
    print("Checking HuggingFace for existing checkpoint...")
    api = HfApi()
    files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)

    for filename in files:
        if filename.startswith("pretrained_models/"):
            print(f"  Skipping {filename}")
            continue
        print(f"  Downloading {filename}...")
        hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=filename,
            repo_type="model",
            token=HF_TOKEN,
            local_dir=SAVE_FOLDER,
        )
    print("Checkpoint restored!")
except RepositoryNotFoundError:
    print("No checkpoint found. Starting fresh.")
except Exception as e:
    print(f"Could not restore: {e}. Starting fresh.")

Checking HuggingFace for existing checkpoint...


.gitattributes: 0.00B [00:00, ?B/s]

CKPT.yaml:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

CKPT+2026-05-16+15-31-02+00/brain.ckpt:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

CKPT+2026-05-16+15-31-02+00/classifier.c(…):   0%|          | 0.00/50.7k [00:00<?, ?B/s]

CKPT+2026-05-16+15-31-02+00/counter.ckpt:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

... (8 outputs left out)

train.json: 0.00B [00:00, ?B/s]

train_log.txt: 0.00B [00:00, ?B/s]

valid.json: 0.00B [00:00, ?B/s]

Checkpoint restored!


In [3]:
# ── Set HF env vars so train.py can access them ───────────────────────
from kaggle_secrets import UserSecretsClient
import os

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"]   = HF_TOKEN
os.environ["HF_REPO_ID"] = "space-cadet/wavlm-ser"
print("HF env vars set")


HF env vars set


In [4]:
# Run this on Kaggle
import torch
from pathlib import Path

save_dir = Path("/kaggle/working/emotion_classification/results/ser_wavlm/1986/save")
for ckpt in sorted(save_dir.glob("CKPT+*")):
    clf = ckpt / "classifier.ckpt"
    if clf.exists():
        state = torch.load(str(clf), map_location="cpu")
        print(ckpt.name, "→", list(state.keys()))

CKPT+2026-05-16+15-31-02+00 → ['w.weight', 'w.bias']


In [5]:
# Install
!pip install speechbrain transformers datasets huggingface_hub==0.32.0 torchaudio --quiet

import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.3/509.3 kB 11.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 53.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 20.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 49.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 21.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 11.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
#  Verify
import torch
import torchaudio
import speechbrain
from transformers import WavLMModel, AutoProcessor

print(f"PyTorch:      {torch.__version__}")
print(f"torchaudio:   {torchaudio.__version__}")
print(f"SpeechBrain:  {speechbrain.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: ")

2026-06-09 18:29:31.837985: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781029772.061221      40 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781029772.124195      40 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


PyTorch:      2.6.0+cu124
torchaudio:   2.6.0+cu124
SpeechBrain:  1.1.0
CUDA:         True
GPU: 


In [7]:
# Load WavLM baseline (downloads ~380MB)
from transformers import WavLMModel, Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-large")
model = WavLMModel.from_pretrained("microsoft/wavlm-large")

print(f"WavLM-Large loaded")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Layers:     {model.config.num_hidden_layers}")
print(f"Hidden dim: {model.config.hidden_size}")
print(f"Attn heads: {model.config.num_attention_heads}")

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

WavLM-Large loaded
Parameters: 315,453,120
Layers:     24
Hidden dim: 1024
Attn heads: 16


In [8]:
!pip install speechbrain==1.0.0 transformers datasets librosa soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.1/760.1 kB 439.0 kB/s eta 0:00:0000:010:01
  Attempting uninstall: speechbrain
    Found existing installation: speechbrain 1.1.0
    Uninstalling speechbrain-1.1.0:
      Successfully uninstalled speechbrain-1.1.0


**Prepare data**

In [9]:
%mkdir -p /kaggle/working/emotion_classification
%cd /kaggle/working/emotion_classification

/kaggle/working/emotion_classification


In [10]:
%%writefile /kaggle/working/emotion_classification/prepare_data.py

#!/usr/bin/env python3
"""
Simple data preparation script for flattened emotion dataset with format:
dataset_emotion_number.wav (e.g., ravdess_happy_0000.wav)
"""

import json
import logging
import librosa
import random
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import Counter

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

class EmotionDatasetProcessor:
    """Processes flattened emotion dataset with dataset_emotion_number.wav format."""
    
    def __init__(self):
        # Valid emotions
        self.valid_emotions = {
            'angry', 'happy', 'sad', 'neutral', 'fearful', 
            'disgust', 'surprised', 'calm'
        }
        
        # Valid datasets
        self.valid_datasets = {
            'ravdess', 'cremad', 'iemocap', 'savee', 'emo-db', 'tess'
        }

        # FIX: Map non-standard labels to valid emotions (recovers ~1852 iemocap_fru files)
        self.emotion_mapping = {
            'fru':         'angry',    # frustration → angry
            'frustration': 'angry',
            'exc':         'happy',    # excitement → happy
            'excitement':  'happy',
            'boredom':     'neutral',
            'bor':         'neutral',
            'oth':         'neutral',  # other → neutral
            'xxx':         None,       # unknown → drop
        }

    def extract_metadata_from_filename(self, filepath: Path) -> dict:
        """Extract dataset and emotion from filename like dataset_emotion_number.wav"""
        filename = filepath.stem.lower()
        
        try:
            parts = filename.split('_')
            
            if len(parts) < 3:
                return None
                
            dataset = parts[0]
            emotion = parts[1]
            
            # Validate dataset
            if dataset not in self.valid_datasets:
                for valid_dataset in self.valid_datasets:
                    if valid_dataset in dataset or dataset in valid_dataset:
                        dataset = valid_dataset
                        break
                else:
                    logger.debug(f"Unknown dataset: {dataset} in {filename}")
                    return None
            
            # Validate / remap emotion
            if emotion not in self.valid_emotions:
                mapped = self.emotion_mapping.get(emotion)
                if mapped is None:
                    logger.debug(f"Unknown emotion: {emotion} in {filename}")
                    return None
                emotion = mapped
            
            return {
                'dataset': dataset,
                'emotion': emotion,
                'valid': True
            }
            
        except Exception as e:
            logger.debug(f"Failed to parse filename {filename}: {e}")
            return None
    
    def process_files(self, data_folder: Path, use_absolute_paths: bool = True) -> dict:
        """Process all audio files and extract metadata."""
        logger.info(f"Processing files in: {data_folder}")
        
        audio_files = list(data_folder.glob("*.wav"))
        audio_files.extend(data_folder.glob("*.WAV"))
        
        if not audio_files:
            logger.error("No .wav files found!")
            return {}
        
        logger.info(f"Found {len(audio_files)} audio files")
        
        processed_data = {}
        stats = {'datasets': {}, 'emotions': {}, 'durations': []}
        failed_files = []
        invalid_format_files = []
        
        for i, audio_file in enumerate(tqdm(audio_files, desc="Processing files")):
            try:
                metadata = self.extract_metadata_from_filename(audio_file)
                
                if not metadata:
                    invalid_format_files.append(audio_file.name)
                    continue
                
                dataset = metadata['dataset']
                emotion = metadata['emotion']
                
                try:
                    y, sr = librosa.load(str(audio_file), sr=None)
                    duration = librosa.get_duration(y=y, sr=sr)
                    
                    if not (0.1 <= duration <= 30):
                        failed_files.append(f"Invalid duration ({duration:.2f}s): {audio_file.name}")
                        continue
                    
                    wav_path = str(audio_file) if use_absolute_paths else audio_file.name
                    entry_id = f"utt-{i:06d}"
                    
                    processed_data[entry_id] = {
                        "id": entry_id,
                        "wav": wav_path,
                        "emotion": emotion,
                        "duration": round(duration, 3),
                        "sample_rate": sr,
                        "dataset": dataset,
                        "filename": audio_file.name
                    }
                    
                    stats['datasets'][dataset] = stats['datasets'].get(dataset, 0) + 1
                    stats['emotions'][emotion] = stats['emotions'].get(emotion, 0) + 1
                    stats['durations'].append(duration)
                    
                except Exception as e:
                    failed_files.append(f"Audio error {audio_file.name}: {str(e)}")
                    continue
                    
            except Exception as e:
                failed_files.append(f"Processing error {audio_file.name}: {str(e)}")
                continue
        
        logger.info(f"\nProcessing Results:")
        logger.info(f"Successfully processed: {len(processed_data)} files")
        logger.info(f"Invalid filename format: {len(invalid_format_files)} files")
        logger.info(f"Failed to load: {len(failed_files)} files")
        logger.info(f"Total files found: {len(audio_files)}")
        
        logger.info(f"\n Dataset Distribution:")
        dataset_counts = sorted(stats['datasets'].items(), key=lambda x: x[1], reverse=True)
        for dataset, count in dataset_counts:
            percentage = count / len(processed_data) * 100
            logger.info(f"  {dataset.upper()}: {count} files ({percentage:.1f}%)")
        
        logger.info(f"\n Emotion Distribution:")
        emotion_counts = sorted(stats['emotions'].items(), key=lambda x: x[1], reverse=True)
        for emotion, count in emotion_counts:
            percentage = count / len(processed_data) * 100
            logger.info(f"  {emotion}: {count} samples ({percentage:.1f}%)")
        
        if stats['durations']:
            durations = stats['durations']
            logger.info(f"\n⏱️ Duration Statistics:")
            logger.info(f"  Average: {sum(durations)/len(durations):.2f}s")
            logger.info(f"  Min: {min(durations):.2f}s")
            logger.info(f"  Max: {max(durations):.2f}s")
        
        if invalid_format_files:
            logger.info(f"\n Sample files with invalid format (expected: dataset_emotion_number.wav):")
            for filename in invalid_format_files[:10]:
                logger.info(f"  {filename}")
            if len(invalid_format_files) > 10:
                logger.info(f"  ... and {len(invalid_format_files)-10} more")
        
        if failed_files:
            logger.info(f"\n Sample failed files:")
            for error in failed_files[:5]:
                logger.info(f"  {error}")
            if len(failed_files) > 5:
                logger.info(f"  ... and {len(failed_files)-5} more")
        
        return processed_data


def create_splits(data_items: list, split_ratio: list) -> dict:
    """Create train/valid/test splits with stratification by emotion."""
    emotions = [item['emotion'] for item in data_items]
    
    try:
        train_size = split_ratio[0] / 100
        temp_size = (split_ratio[1] + split_ratio[2]) / 100
        
        train_data, temp_data = train_test_split(
            data_items, test_size=temp_size, stratify=emotions, random_state=42
        )
        
        valid_ratio = split_ratio[1] / (split_ratio[1] + split_ratio[2])
        temp_emotions = [item['emotion'] for item in temp_data]
        
        valid_data, test_data = train_test_split(
            temp_data, test_size=(1 - valid_ratio), stratify=temp_emotions, random_state=42
        )
        
        logger.info("Used stratified split by emotion")
        return {'train': train_data, 'valid': valid_data, 'test': test_data}
        
    except ValueError as e:
        logger.warning(f"Stratified split failed: {e}. Using random split.")
        random.shuffle(data_items)
        total = len(data_items)
        train_end = int(total * split_ratio[0] / 100)
        valid_end = train_end + int(total * split_ratio[1] / 100)
        return {
            'train': data_items[:train_end],
            'valid': data_items[train_end:valid_end],
            'test': data_items[valid_end:]
        }


def save_data_files(split_data: dict, save_path: Path, unique_emotions: list):
    """Save JSON files and label encoder for SpeechBrain."""
    for split_name, split_items in split_data.items():
        output_file = save_path / f"{split_name}.json"
        json_data = {}
        for item in split_items:
            json_data[item["id"]] = {
                "wav": item["wav"],
                "emotion": item["emotion"]
            }
        with open(output_file, 'w') as f:
            json.dump(json_data, f, indent=2)
        logger.info(f"Created {split_name}.json with {len(split_items)} samples")
    
    label_file = save_path / "label_encoder.txt"
    with open(label_file, 'w') as f:
        for emotion in unique_emotions:
            f.write(f"{emotion}\n")
    logger.info(f"Created label_encoder.txt with {len(unique_emotions)} emotions")


def prepare_data(data_folder: str, save_folder: str, splits: list,
                 split_ratio: list, use_absolute_paths: bool = True, force: bool = False):
    logger.info("Starting data preparation for dataset_emotion_number.wav format...")
    
    data_folder = Path(data_folder).resolve()
    save_folder = Path(save_folder).resolve()
    
    logger.info(f"Data folder: {data_folder}")
    logger.info(f"Save folder: {save_folder}")
    
    if not data_folder.exists():
        raise FileNotFoundError(f"Data folder does not exist: {data_folder}")
    
    save_path = save_folder / "save"
    save_path.mkdir(parents=True, exist_ok=True)
    
    required_files = [
        save_path / "train.json",
        save_path / "valid.json",
        save_path / "test.json",
        save_path / "label_encoder.txt"
    ]
    
    if not force and all(f.exists() for f in required_files):
        logger.info("Data already prepared. Use force=True to recreate.")
        return True
    
    processor = EmotionDatasetProcessor()
    processed_data = processor.process_files(data_folder, use_absolute_paths)
    
    if not processed_data:
        raise RuntimeError("No valid samples found! Check your file naming format.")
    
    if len(processed_data) < 15000:
        logger.warning(f" Only processed {len(processed_data)} files. Expected ~20k.")
    
    data_items = list(processed_data.values())
    emotions = [item['emotion'] for item in data_items]
    unique_emotions = sorted(set(emotions))
    
    logger.info(f"Creating splits for {len(data_items)} samples")
    logger.info(f"Unique emotions: {unique_emotions}")
    
    split_data = create_splits(data_items, split_ratio)
    save_data_files(split_data, save_path, unique_emotions)
    
    logger.info(f"\nData preparation completed!")
    logger.info(f"Final split sizes:")
    total_samples = sum(len(split_items) for split_items in split_data.values())
    
    for split_name, split_items in split_data.items():
        percentage = len(split_items) / total_samples * 100
        logger.info(f"  {split_name}: {len(split_items)} samples ({percentage:.1f}%)")
    
    logger.info(f"\nEmotion distribution per split:")
    for split_name, split_items in split_data.items():
        split_emotions = Counter(item['emotion'] for item in split_items)
        emotion_dist = ", ".join([f"{emotion}: {count}" for emotion, count in split_emotions.most_common()])
        logger.info(f"  {split_name}: {emotion_dist}")
    
    all_datasets = Counter(item['dataset'] for item in data_items)
    logger.info(f"\nDataset distribution across all splits:")
    for dataset, count in all_datasets.most_common():
        percentage = count / len(data_items) * 100
        logger.info(f"  {dataset.upper()}: {count} files ({percentage:.1f}%)")
    
    return True


if __name__ == "__main__":
    DATA_FOLDER = "/kaggle/input/processed-ser-data"
    SAVE_FOLDER = "/kaggle/working/emotion_classification/results/ser_wavlm/1986"
    SPLITS = ["train", "valid", "test"]
    SPLIT_RATIO = [80, 10, 10]
    
    prepare_data(
        data_folder=DATA_FOLDER,
        save_folder=SAVE_FOLDER,
        splits=SPLITS,
        split_ratio=SPLIT_RATIO,
        use_absolute_paths=True,
        force=True
    )

Writing /kaggle/working/emotion_classification/prepare_data.py


In [11]:
%cd /kaggle/working/emotion_classification
!python prepare_data.py

/kaggle/working/emotion_classification
Processing files: 100%|██████████████████| 17854/17854 [02:49<00:00, 105.27it/s]


**Train Classifier**

In [12]:
%%writefile /kaggle/working/emotion_classification/train.yaml

seed: 1986
__set_torchseed: !apply:torch.manual_seed [!ref <seed>]
output_folder: !ref results/ser_wavlm/<seed>
save_folder: !ref <output_folder>/save
train_log: !ref <output_folder>/train_log.txt

# ── DATA ──────────────────────────────────────────────────────────────
data_folder: /kaggle/input/processed-ser-data/
label_encoder_file: !ref <save_folder>/label_encoder.txt
train_json: !ref <save_folder>/train.json
valid_json: !ref <save_folder>/valid.json
test_json:  !ref <save_folder>/test.json

# ── AUDIO ─────────────────────────────────────────────────────────────
sample_rate: 16000

# ── MODEL DIMENSIONS ──────────────────────────────────────────────────
embedding_dim: 768
pooled_dim: 1536
n_classes: 8
dropout: 0.3

# ── WavLM BACKBONE ────────────────────────────────────────────────────
wavlm: !new:speechbrain.lobes.models.huggingface_transformers.wavlm.WavLM
    source: microsoft/wavlm-base-plus
    save_path: pretrained_models/wavlm
    output_norm: True
    freeze: False
    freeze_feature_extractor: True

# ── POOLING ───────────────────────────────────────────────────────────
pooling: !new:speechbrain.nnet.pooling.StatisticsPooling

# ── CLASSIFIER ────────────────────────────────────────────────────────
# Placeholder only — train.py replaces this with a proper MLP at runtime.
# We keep it as a simple Linear so the YAML parses without errors.
classifier: !new:train.MLPClassifier
    input_size: !ref <pooled_dim>
    hidden: 256
    n_classes: !ref <n_classes>
    dropout: !ref <dropout>

# ── NORMALISATION ─────────────────────────────────────────────────────
normalize: !new:speechbrain.processing.features.InputNormalization
    norm_type: global
    std_norm: False

# ── MODULES ───────────────────────────────────────────────────────────
modules:
    normalize: !ref <normalize>
    wavlm: !ref <wavlm>
    pooling: !ref <pooling>
    classifier: !ref <classifier>

# ── LOSS ──────────────────────────────────────────────────────────────
compute_cost: !name:speechbrain.nnet.losses.nll_loss

error_stats: !name:speechbrain.utils.metric_stats.MetricStats
    metric: !name:speechbrain.nnet.losses.classification_error
        reduction: batch

# ── TRAINING ──────────────────────────────────────────────────────────
number_of_epochs: 40
batch_size: 4
lr: 0.000005
lr_final: 0.0000001

# ── OPTIMISER ─────────────────────────────────────────────────────────
opt_class: !name:torch.optim.AdamW
    lr: !ref <lr>
    weight_decay: 0.05

# ── LR SCHEDULER ─────────────────────────────────────────────────────
lr_annealing: !new:speechbrain.nnet.schedulers.NewBobScheduler
    initial_value: !ref <lr>
    improvement_threshold: 0.0025
    annealing_factor: 0.8
    patient: 2

# ── LOGGING ───────────────────────────────────────────────────────────
train_logger: !new:speechbrain.utils.train_logger.FileTrainLogger
    save_file: !ref <train_log>

# ── DATALOADERS ───────────────────────────────────────────────────────
train_dataloader_opts:
    batch_size: !ref <batch_size>
    shuffle: True
    num_workers: 2

valid_dataloader_opts:
    batch_size: !ref <batch_size>
    num_workers: 2

test_dataloader_opts:
    batch_size: !ref <batch_size>
    num_workers: 2

# ── PRECISION ─────────────────────────────────────────────────────────
precision: fp16

# ── CHECKPOINTING ─────────────────────────────────────────────────────
epoch_counter: !new:speechbrain.utils.epoch_loop.EpochCounter
    limit: !ref <number_of_epochs>

checkpointer: !new:speechbrain.utils.checkpoints.Checkpointer
    checkpoints_dir: !ref <save_folder>
    recoverables:
        wavlm: !ref <wavlm>
        pooling: !ref <pooling>
        classifier: !ref <classifier>
        normalizer: !ref <normalize>
        counter: !ref <epoch_counter>

Writing /kaggle/working/emotion_classification/train.yaml


In [1]:
%%writefile /kaggle/working/emotion_classification/train.py
#!/usr/bin/env python3
import os
import sys
import glob
import shutil
import random
import torch
import torchaudio
import logging
import speechbrain as sb
from speechbrain.utils.distributed import run_on_main
from hyperpyyaml import load_hyperpyyaml
from collections import Counter

logger = logging.getLogger(__name__)
EARLY_STOP_PATIENCE = 5


# ── MLP Classifier (SpeechBrain-compatible for checkpointing) ──────────
class MLPClassifier(torch.nn.Module):
    def __init__(self, input_size, hidden=256, n_classes=8, dropout=0.3):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_size, hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        if x.dim() == 3:
            x = x.squeeze(1)
        return self.net(x)


class EmotionBrain(sb.core.Brain):

    def on_fit_start(self):
        if not hasattr(self, "optimizer"):
            self.init_optimizers()

    def compute_forward(self, batch, stage):
        batch = batch.to(self.device)
        wavs, wav_lens = batch.sig
        emotions, _ = batch.emo_encoded

        if stage == sb.Stage.TRAIN:
            speed_factor = random.choice([0.9, 1.0, 1.1])
            if speed_factor != 1.0:
                wavs = torchaudio.functional.resample(
                    wavs,
                    orig_freq=int(16000 * speed_factor),
                    new_freq=16000,
                )
            if random.random() < 0.5:
                noise = torch.randn_like(wavs) * 0.005
                wavs = wavs + noise

        features = self.modules.wavlm(wavs)
        if isinstance(features, dict):
            features = features["last_hidden_state"]

        pooled = self.modules.pooling(features, wav_lens)
        predictions = self.modules.classifier(pooled)
        return predictions, emotions

    def compute_objectives(self, predictions, targets, stage):
        predictions, emotions = predictions

        if not hasattr(self, "_debug_printed"):
            print(f"  Predictions shape: {predictions.shape}")
            print(f"  Targets shape:     {emotions.shape}")
            self._debug_printed = True

        log_probs = torch.log_softmax(predictions, dim=-1)

        loss = self.hparams.compute_cost(
            log_probs,
            emotions.long().squeeze(-1),
            weight=self.hparams.class_weights.to(self.device),
        )

        if stage != sb.Stage.TRAIN:
            # ── Manual accuracy tracking for clean 0–1 error rate ──────
            preds_idx   = predictions.argmax(dim=-1)   # [B]
            targets_idx = emotions.squeeze(-1)          # [B]
            self._total_correct += (preds_idx == targets_idx).sum().item()
            self._total_samples += targets_idx.shape[0]

        return loss

    def on_stage_start(self, stage, epoch):
        if stage != sb.Stage.TRAIN:
            self._total_correct = 0
            self._total_samples = 0

    def on_stage_end(self, stage, stage_loss, epoch):
        stage_stats = {"loss": stage_loss}

        if stage == sb.Stage.TRAIN:
            self.train_stats = stage_stats

        elif stage == sb.Stage.VALID:
            # ── Proper 0–1 error rate (1 = all wrong, 0 = all correct) ─
            error_rate = 1.0 - (self._total_correct / max(self._total_samples, 1))
            stage_stats["error_rate"] = error_rate
            accuracy   = self._total_correct / max(self._total_samples, 1)
            print(f"  Accuracy: {accuracy:.4f} | Error rate: {error_rate:.4f} "
                  f"({self._total_correct}/{self._total_samples} correct)")

            old_lr, new_lr = self.hparams.lr_annealing(stage_stats["error_rate"])
            sb.nnet.schedulers.update_learning_rate(self.optimizer, new_lr)

            self.hparams.train_logger.log_stats(
                stats_meta={"epoch": epoch, "lr": old_lr},
                train_stats=self.train_stats,
                valid_stats=stage_stats,
            )

            self.checkpointer.save_and_keep_only(
                meta={"error_rate": stage_stats["error_rate"]},
                min_keys=["error_rate"],
            )

            # ── Push checkpoint to HuggingFace after every epoch ───────
            try:
                from huggingface_hub import HfApi
                _hf_api   = HfApi()
                _hf_repo  = os.environ.get("HF_REPO_ID", "space-cadet/wavlm-ser").strip()
                _hf_token = os.environ.get("HF_TOKEN", None)
                _save_dir = self.hparams.save_folder

                # ── Copy train log into save folder so it gets pushed ──
                _log_src = self.hparams.train_log
                _log_dst = os.path.join(_save_dir, "train_log.txt")
                if os.path.exists(_log_src):
                    shutil.copy2(_log_src, _log_dst)

                # ── 1. List old CKPT folders BEFORE uploading ──────────
                existing_files = list(_hf_api.list_repo_files(
                    repo_id=_hf_repo, repo_type="model", token=_hf_token
                ))
                old_ckpts = sorted(set(
                    f.split("/")[0] for f in existing_files
                    if f.startswith("CKPT+") and "/" in f
                ))

                # ── 2. Upload new checkpoint ───────────────────────────
                _hf_api.upload_folder(
                    folder_path=_save_dir,
                    repo_id=_hf_repo,
                    repo_type="model",
                    token=_hf_token,
                    ignore_patterns=["pretrained_models/**"],
                    commit_message=f"Epoch {epoch} | error_rate: {stage_stats['error_rate']:.4f}",
                )
                print(f"  Epoch {epoch} pushed to HuggingFace ({_hf_repo})")

                # ── 3. Delete old CKPT folders after upload ────────────
                for old in old_ckpts:
                    try:
                        _hf_api.delete_files(
                            repo_id=_hf_repo,
                            repo_type="model",
                            delete_patterns=[f"{old}/**"],
                            token=_hf_token,
                            commit_message=f"Remove old checkpoint {old}",
                        )
                        print(f"    Deleted old checkpoint {old}")
                    except Exception as _del_err:
                        print(f"  Could not delete {old}: {_del_err}")

            except Exception as _hf_err:
                print(f"  HF push failed (epoch {epoch}): {_hf_err}")

            # ── Early stopping ─────────────────────────────────────────
            current_err = stage_stats["error_rate"]
            if not hasattr(self, "_best_error_rate"):
                self._best_error_rate = current_err
                self._epochs_no_improve = 0
            elif current_err < self._best_error_rate - 0.001:
                print(f"  Improved: {self._best_error_rate:.4f} → {current_err:.4f}")
                self._best_error_rate = current_err
                self._epochs_no_improve = 0
            else:
                self._epochs_no_improve += 1
                print(
                    f"  No improvement for {self._epochs_no_improve}/{EARLY_STOP_PATIENCE} epoch(s). "
                    f"Best: {self._best_error_rate:.4f}, Current: {current_err:.4f}"
                )
                if self._epochs_no_improve >= EARLY_STOP_PATIENCE:
                    print("  Early stopping triggered.")
                    self.hparams.epoch_counter.current = self.hparams.epoch_counter.limit

        elif stage == sb.Stage.TEST:
            error_rate = 1.0 - (self._total_correct / max(self._total_samples, 1))
            accuracy   = self._total_correct / max(self._total_samples, 1)
            stage_stats["error_rate"] = error_rate
            print(f"  Test Accuracy: {accuracy:.4f} | Error rate: {error_rate:.4f} "
                  f"({self._total_correct}/{self._total_samples} correct)")
            self.hparams.train_logger.log_stats(
                stats_meta={"Epoch loaded": self.hparams.epoch_counter.current},
                test_stats=stage_stats,
            )


# ── Safe checkpoint recovery helper ───────────────────────────────────
def safe_recover_checkpoint(brain):
    try:
        brain.checkpointer.recover_if_possible()
        print("   Checkpoint recovered successfully.")
    except (KeyError, RuntimeError) as e:
        print(f"   Checkpoint incompatible ({e}). Clearing stale checkpoints and starting fresh.")
        
        # ── Clear local checkpoints ────────────────────────────────
        ckpt_dir = brain.hparams.save_folder
        removed = 0
        for ckpt_path in glob.glob(os.path.join(ckpt_dir, "CKPT+*")):
            shutil.rmtree(ckpt_path, ignore_errors=True)
            print(f"    Removed local: {ckpt_path}")
            removed += 1
        if removed == 0:
            print("    No local checkpoint directories found.")

        # ── Clear HuggingFace checkpoints ─────────────────────────
        try:
            from huggingface_hub import HfApi
            _hf_api   = HfApi()
            _hf_repo  = os.environ.get("HF_REPO_ID", "space-cadet/wavlm-ser").strip()
            _hf_token = os.environ.get("HF_TOKEN", None)

            existing_files = list(_hf_api.list_repo_files(
                repo_id=_hf_repo, repo_type="model", token=_hf_token
            ))
            old_ckpts = sorted(set(
                f.split("/")[0] for f in existing_files
                if f.startswith("CKPT+") and "/" in f
            ))
            for old in old_ckpts:
                _hf_api.delete_files(
                    repo_id=_hf_repo,
                    repo_type="model",
                    delete_patterns=[f"{old}/**"],
                    token=_hf_token,
                    commit_message=f"Auto-removed incompatible checkpoint {old}",
                )
                print(f"    Deleted HF checkpoint: {old}")
            if not old_ckpts:
                print("    No HF checkpoints found to delete.")
        except Exception as hf_err:
            print(f"    HF cleanup failed: {hf_err}")

        print("  Starting training from scratch (epoch 1).")


# ── DATA PIPELINE ──────────────────────────────────────────────────────
def dataio_prepare(hparams):
    data_folder = hparams["data_folder"]

    train_data = sb.dataio.dataset.DynamicItemDataset.from_json(
        json_path=hparams["train_json"],
        replacements={"data_root": data_folder},
    )
    valid_data = sb.dataio.dataset.DynamicItemDataset.from_json(
        json_path=hparams["valid_json"],
        replacements={"data_root": data_folder},
    )
    test_data = sb.dataio.dataset.DynamicItemDataset.from_json(
        json_path=hparams["test_json"],
        replacements={"data_root": data_folder},
    )

    datasets = [train_data, valid_data, test_data]

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline(wav):
        return sb.dataio.dataio.read_audio(wav)

    sb.dataio.dataset.add_dynamic_item(datasets, audio_pipeline)

    print("Scanning dataset for labels...")
    all_emotions = set()
    for dataset in datasets:
        for item in dataset.data.values():
            if "emotion" in item:
                all_emotions.add(item["emotion"])

    emotions_list = sorted(list(all_emotions))
    num_classes   = len(emotions_list)
    print(f" Classes ({num_classes}): {emotions_list}")

    label_encoder = sb.dataio.encoder.CategoricalEncoder()
    for emo in emotions_list:
        label_encoder.add_label(emo)
    label_encoder.expect_len(num_classes)

    pooled_dim = hparams["pooled_dim"]
    dropout    = hparams.get("dropout", 0.3)
    print(f"Updating classifier → input: {pooled_dim}, output: {num_classes}, dropout: {dropout}")

    train_counts  = Counter(item["emotion"] for item in train_data.data.values())
    total         = sum(train_counts.values())
    class_weights = torch.tensor(
        [total / (num_classes * train_counts.get(e, 1)) for e in emotions_list],
        dtype=torch.float,
    )
    hparams["class_weights"] = class_weights
    print(f"Class weights: { {e: f'{w:.2f}' for e, w in zip(emotions_list, class_weights)} }")

    @sb.utils.data_pipeline.takes("emotion")
    @sb.utils.data_pipeline.provides("emotion", "emo_encoded")
    def emotion_pipeline(emotion):
        emo_encoded = label_encoder.encode_label_torch(emotion)
        if emo_encoded.dim() == 0:
            emo_encoded = emo_encoded.unsqueeze(0)
        yield emotion
        yield emo_encoded.long()

    sb.dataio.dataset.add_dynamic_item(datasets, emotion_pipeline)

    os.makedirs(os.path.dirname(hparams["label_encoder_file"]), exist_ok=True)
    label_encoder.save(hparams["label_encoder_file"])

    sb.dataio.dataset.set_output_keys(datasets, ["id", "sig", "emo_encoded"])

    return train_data, valid_data, test_data, label_encoder


# ── MAIN ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    _clean_argv = [a for a in sys.argv[1:] if not a.startswith("-") or a.endswith(".yaml")]
    if not any(a.endswith(".yaml") for a in _clean_argv):
        _clean_argv = ["/kaggle/working/emotion_classification/train.yaml"]

    hparams_file, run_opts, overrides = sb.parse_arguments(_clean_argv)
    sb.utils.distributed.ddp_init_group(run_opts)

    with open(hparams_file) as fin:
        hparams = load_hyperpyyaml(fin, overrides)

    sb.create_experiment_directory(
        experiment_directory=hparams["output_folder"],
        hyperparams_to_save=hparams_file,
        overrides=overrides,
    )

    from prepare_data import prepare_data

    run_on_main(
        prepare_data,
        kwargs={
            "data_folder":  hparams["data_folder"],
            "save_folder":  hparams["output_folder"],
            "splits":       ["train", "valid", "test"],
            "split_ratio":  [80, 10, 10],
        },
    )

    train_data, valid_data, test_data, label_encoder = dataio_prepare(hparams)

    print(" Building model...")
    emotion_brain = EmotionBrain(
        modules=hparams["modules"],
        opt_class=hparams["opt_class"],
        hparams=hparams,
        run_opts=run_opts,
        checkpointer=hparams["checkpointer"],
    )

    print(f"Params: {sum(p.numel() for p in emotion_brain.modules.parameters())/1e6:.1f}M")
    print("Recovering checkpoint (if any)...")
    safe_recover_checkpoint(emotion_brain)

    print("Training...")
    emotion_brain.fit(
        emotion_brain.hparams.epoch_counter,
        train_data,
        valid_data,
        train_loader_kwargs=hparams["train_dataloader_opts"],
        valid_loader_kwargs=hparams["valid_dataloader_opts"],
    )

    print(" Testing...")
    emotion_brain.evaluate(
        test_data,
        test_loader_kwargs=hparams["test_dataloader_opts"],
    )

Overwriting /kaggle/working/emotion_classification/train.py


In [14]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
%cd /kaggle/working/emotion_classification
!python train.py train.yaml

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
/kaggle/working/emotion_classification
2026-06-09 18:41:37.612256: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781030497.636497     333 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781030497.644305     333 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
speechbrain.core - Beginning experiment!
speechbrain.core - Experiment folder: results/ser_wavlm/1986
prepare_data - Starting data preparation for dataset_emotion_number.wav format...
prepare_data - Data folder: /kaggle/input/processed-ser-data
prepare_data - Save folder: /kaggle/working/emotion_classification/results/ser_wavlm/1986
prepare_d

In [16]:
# Create folder for best model
!mkdir /kaggle/working/best_model/

# Copy label encoder
!cp /kaggle/working/emotion_classification/results/emotion_recognition/1986/save/label_encoder.txt /kaggle/working/best_model

# Copy best model
!cp "`ls -td results/ser_wavlm/1986/save/CKPT* | head -1`"/* /kaggle/working/best_model

cp: cannot stat '/kaggle/working/emotion_classification/results/emotion_recognition/1986/save/label_encoder.txt': No such file or directory


**Use classifier**

In [17]:
%%writefile /kaggle/working/best_model/hparams_inference.yaml

# ##############################################
# Emotion Recognition Inference (WavLM)
# ##############################################

pretrained_path: /kaggle/working/best_model/

# --------------------------------------------
# Audio
# --------------------------------------------
sample_rate: 16000
n_classes: 8
embedding_dim: 768
pooled_dim: 1536   # embedding_dim * 2

# --------------------------------------------
# WavLM Backbone
# FIX: was "wavlm-base", must match training: "wavlm-base-plus"
# --------------------------------------------
wavlm: !new:speechbrain.lobes.models.huggingface_transformers.wavlm.WavLM
    source: microsoft/wavlm-base-plus
    save_path: !ref <pretrained_path>/wavlm_cache
    output_norm: True
    freeze: True

# --------------------------------------------
# Pooling
# FIX: was AttentiveStatisticsPooling, must match training: StatisticsPooling
# --------------------------------------------
pooling: !new:speechbrain.nnet.pooling.StatisticsPooling

# --------------------------------------------
# Classifier
# --------------------------------------------
classifier: !new:speechbrain.nnet.linear.Linear
    input_size: !ref <pooled_dim>
    n_neurons: !ref <n_classes>

# --------------------------------------------
# Normalisation
# --------------------------------------------
normalize: !new:speechbrain.processing.features.InputNormalization
    norm_type: global
    std_norm: False

# --------------------------------------------
# Label encoder
# --------------------------------------------
label_encoder: !new:speechbrain.dataio.encoder.CategoricalEncoder

# --------------------------------------------
# Modules
# --------------------------------------------
modules:
    normalize: !ref <normalize>
    wavlm: !ref <wavlm>
    pooling: !ref <pooling>
    classifier: !ref <classifier>

# --------------------------------------------
# Load pretrained weights
# --------------------------------------------
pretrainer: !new:speechbrain.utils.parameter_transfer.Pretrainer
    loadables:
        wavlm: !ref <wavlm>
        classifier: !ref <classifier>
        label_encoder: !ref <label_encoder>

    paths:
        wavlm: !ref <pretrained_path>/wavlm.ckpt
        classifier: !ref <pretrained_path>/classifier.ckpt
        label_encoder: !ref <pretrained_path>/label_encoder.txt

Writing /kaggle/working/best_model/hparams_inference.yaml


In [ ]:
import os
import glob
import torch
import torchaudio
import torch.nn as nn
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from speechbrain.dataio.encoder import CategoricalEncoder

# ── CONFIG ────────────────────────────────────────────────────────────
CHECKPOINT_DIR     = "/kaggle/working/emotion_classification/results/ser_wavlm/1986/save"
LABEL_ENCODER_FILE = os.path.join(CHECKPOINT_DIR, "label_encoder.txt")
DEVICE             = "cuda" if torch.cuda.is_available() else "cpu"

HIDDEN_DIM  = 768           # wavlm-base-plus
POOLED_DIM  = HIDDEN_DIM * 2
N_CLASSES   = 8
HF_MODEL_ID = "microsoft/wavlm-base-plus"

# ── 1. LOAD WavLM VIA HUGGINGFACE (no SpeechBrain import tricks) ──────
print(f"Loading {HF_MODEL_ID}...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(HF_MODEL_ID)
wavlm = WavLMModel.from_pretrained(HF_MODEL_ID).to(DEVICE)
wavlm.eval()
print(f" WavLM loaded | hidden: {wavlm.config.hidden_size}")

# ── 2. STATISTICS POOLING (matches SpeechBrain exactly) ───────────────
class StatisticsPooling(nn.Module):
    def forward(self, x):           # x: [B, T, H]
        mean = x.mean(dim=1)
        std  = x.std(dim=1)
        return torch.cat([mean, std], dim=-1)   # [B, H*2]

# ── 3. CLASSIFIER ─────────────────────────────────────────────────────
#   If you applied the anti-overfitting fix (train.py with 2-layer MLP),
#     use ClassifierMLP below. Otherwise use ClassifierLinear.
#
#     To check: open your best checkpoint folder and look at classifier.ckpt.
#     If it has keys like '0.weight','2.weight','3.weight' → MLP
#     If it has keys like 'w.weight' → Linear
#
# ── Option A: original single Linear (matches old train.yaml) ─────────
class ClassifierLinear(nn.Module):
    def __init__(self, input_size, n_classes):
        super().__init__()
        self.w = nn.Linear(input_size, n_classes, bias=True)
    def forward(self, x):
        return self.w(x)

# ── Option B: 2-layer MLP (matches new anti-overfitting train.py) ─────
class ClassifierMLP(nn.Module):
    def __init__(self, input_size, hidden=256, n_classes=8, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)

pooling    = StatisticsPooling().to(DEVICE)

# ── AUTO-DETECT which classifier to use from checkpoint keys ──────────
def build_classifier(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE)
    keys  = list(state.keys())
    print(f"   Classifier keys: {keys[:6]}")
    if any(k.startswith("w.") for k in keys):
        print("   → Detected: single Linear classifier")
        clf = ClassifierLinear(POOLED_DIM, N_CLASSES)
    else:
        print("   → Detected: MLP classifier")
        clf = ClassifierMLP(POOLED_DIM, hidden=256, n_classes=N_CLASSES)
    clf.load_state_dict(state)
    return clf.to(DEVICE)

# ── 4. FIND BEST CHECKPOINT ───────────────────────────────────────────
ckpt_folders = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "CKPT+*")))
assert ckpt_folders, f"No checkpoints found in {CHECKPOINT_DIR}"

# SpeechBrain keeps only the best checkpoint (save_and_keep_only)
# so there should be exactly one folder here
best_ckpt = ckpt_folders[0]   # head-1 = most recently saved = best
print(f"Checkpoint: {best_ckpt}")

# ── 5. LOAD CLASSIFIER ────────────────────────────────────────────────
classifier_path = os.path.join(best_ckpt, "classifier.ckpt")
assert os.path.exists(classifier_path), f"Missing: {classifier_path}"
classifier = build_classifier(classifier_path)
classifier.eval()

# ── 6. LOAD WavLM WEIGHTS ─────────────────────────────────────────────
wavlm_ckpt_path = os.path.join(best_ckpt, "wavlm.ckpt")
assert os.path.exists(wavlm_ckpt_path), f"Missing: {wavlm_ckpt_path}"
wavlm_state = torch.load(wavlm_ckpt_path, map_location=DEVICE)

# SpeechBrain sometimes wraps weights under a 'model.' prefix — strip it
if any(k.startswith("model.") for k in wavlm_state.keys()):
    wavlm_state = {k.replace("model.", "", 1): v for k, v in wavlm_state.items()}

missing, unexpected = wavlm.load_state_dict(wavlm_state, strict=False)
if missing:
    print(f"   Missing keys in WavLM: {missing[:5]}")
if unexpected:
    print(f"   Unexpected keys in WavLM: {unexpected[:5]}")
print("   loaded wavlm.ckpt")

# pooling.ckpt is empty for StatisticsPooling (no learned params) — skip it
wavlm.eval()

# ── 7. LABEL ENCODER ─────────────────────────────────────────────────
label_encoder = CategoricalEncoder()
label_encoder.load(LABEL_ENCODER_FILE)
idx2label = label_encoder.ind2lab
print(f"Classes: {idx2label}")

# ── 8. INFERENCE ──────────────────────────────────────────────────────
def predict_file(audio_path):
    signal, fs = torchaudio.load(audio_path)

    if fs != 16000:
        signal = torchaudio.functional.resample(signal, fs, 16000)
    if signal.shape[0] > 1:
        signal = signal.mean(dim=0, keepdim=True)   # stereo → mono

    inputs = feature_extractor(
        signal.squeeze(0).numpy(),
        sampling_rate=16000,
        return_tensors="pt",
        padding=True,
    )
    input_values = inputs.input_values.to(DEVICE)

    with torch.no_grad():
        outputs  = wavlm(input_values)
        features = outputs.last_hidden_state         # [1, T, H]
        pooled   = pooling(features)                 # [1, POOLED_DIM]
        logits   = classifier(pooled)                # [1, N_CLASSES]
        probs    = torch.softmax(logits, dim=-1)
        conf, idx = probs.max(dim=-1)

    return {
        "file":       os.path.basename(audio_path),
        "prediction": idx2label[idx.item()],
        "confidence": round(conf.item(), 4),
        "all_probs":  {idx2label[i]: round(p.item(), 3)
                       for i, p in enumerate(probs.squeeze())},
    }

# ── 9. RUN ────────────────────────────────────────────────────────────
audio_files = glob.glob("/kaggle/input/test-data/*.wav")[:10]
assert audio_files, "No .wav files found — update the path above"

for f in audio_files:
    result = predict_file(f)
    print(f"\n{result['file']}")
    print(f"  → {result['prediction']} ({result['confidence']:.1%})")
    print(f"  all: {result['all_probs']}")

Loading microsoft/wavlm-base-plus...


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

 WavLM loaded | hidden: 768
Checkpoint: /kaggle/working/emotion_classification/results/ser_wavlm/1986/save/CKPT+2026-05-16+15-31-02+00
   Classifier keys: ['w.weight', 'w.bias']
   → Detected: single Linear classifier
   loaded wavlm.ckpt
Classes: {0: 'angry', 1: 'calm', 2: 'disgust', 3: 'fearful', 4: 'happy', 5: 'neutral', 6: 'sad', 7: 'surprised'}


In [ ]:
%%writefile /kaggle/working/emotion_classification/evaluate.py
#!/usr/bin/env python3
"""
Run after training. Produces:
  1. Confusion matrix (saved as PNG + shown inline)
  2. Per-sample predictions CSV
  3. Per-class accuracy summary
"""

import os
import glob
import json
import random
import torch
import torchaudio
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from speechbrain.dataio.encoder import CategoricalEncoder

# ── CONFIG ────────────────────────────────────────────────────────────
CHECKPOINT_DIR     = "/kaggle/working/emotion_classification/results/ser_wavlm/1986/save"
LABEL_ENCODER_FILE = os.path.join(CHECKPOINT_DIR, "label_encoder.txt")
TEST_JSON          = os.path.join(CHECKPOINT_DIR, "test.json")
OUTPUT_DIR         = "/kaggle/working/emotion_classification/eval_outputs"
DEVICE             = "cuda" if torch.cuda.is_available() else "cpu"

HIDDEN_DIM  = 768
POOLED_DIM  = HIDDEN_DIM * 2
HF_MODEL_ID = "microsoft/wavlm-base-plus"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")
print(f"Device: {DEVICE}")

# ── 1. LOAD LABEL ENCODER ─────────────────────────────────────────────
label_encoder = CategoricalEncoder()
label_encoder.load(LABEL_ENCODER_FILE)
idx2label = label_encoder.ind2lab
label2idx = {v: k for k, v in idx2label.items()}
class_names = [idx2label[i] for i in range(len(idx2label))]
n_classes = len(class_names)
print(f"Classes ({n_classes}): {class_names}")

# ── 2. LOAD WavLM ─────────────────────────────────────────────────────
print(f"\nLoading {HF_MODEL_ID}...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(HF_MODEL_ID)
wavlm = WavLMModel.from_pretrained(HF_MODEL_ID).to(DEVICE)
print(f"WavLM loaded")

# ── 3. STATISTICS POOLING ─────────────────────────────────────────────
class StatisticsPooling(nn.Module):
    def forward(self, x):
        return torch.cat([x.mean(dim=1), x.std(dim=1)], dim=-1)

pooling = StatisticsPooling().to(DEVICE)

# ── 4. AUTO-DETECT & LOAD CLASSIFIER ─────────────────────────────────
ckpt_folders = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "CKPT+*")))
assert ckpt_folders, f"No checkpoints in {CHECKPOINT_DIR}"
best_ckpt = ckpt_folders[0]
print(f"Checkpoint: {best_ckpt}")

classifier_path = os.path.join(best_ckpt, "classifier.ckpt")
assert os.path.exists(classifier_path), f"Missing: {classifier_path}"

state = torch.load(classifier_path, map_location=DEVICE)
keys  = list(state.keys())
print(f"   Classifier keys: {keys[:6]}")

if any(k.startswith("w.") for k in keys):
    # SpeechBrain single Linear (w.weight / w.bias)
    print("   → Single Linear classifier")
    classifier = nn.Linear(POOLED_DIM, n_classes)
    # remap w.weight → weight
    new_state = {k.replace("w.", ""): v for k, v in state.items()}
    classifier.load_state_dict(new_state)
else:
    # torch.nn.Sequential MLP (0.weight, 3.weight …)
    print("   → MLP classifier")
    classifier = nn.Sequential(
        nn.Linear(POOLED_DIM, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, n_classes),
    )
    classifier.load_state_dict(state)

classifier = classifier.to(DEVICE)

# ── 5. LOAD WavLM WEIGHTS ─────────────────────────────────────────────
wavlm_state = torch.load(os.path.join(best_ckpt, "wavlm.ckpt"), map_location=DEVICE)
if any(k.startswith("model.") for k in wavlm_state):
    wavlm_state = {k.replace("model.", "", 1): v for k, v in wavlm_state.items()}
wavlm.load_state_dict(wavlm_state, strict=False)
wavlm.eval(); pooling.eval(); classifier.eval()
print("All weights loaded\n")

# ── 6. INFERENCE FUNCTION ─────────────────────────────────────────────
def predict_file(audio_path):
    signal, fs = torchaudio.load(audio_path)
    if fs != 16000:
        signal = torchaudio.functional.resample(signal, fs, 16000)
    if signal.shape[0] > 1:
        signal = signal.mean(dim=0, keepdim=True)

    inputs = feature_extractor(
        signal.squeeze(0).numpy(),
        sampling_rate=16000,
        return_tensors="pt",
        padding=True,
    )
    with torch.no_grad():
        features = wavlm(inputs.input_values.to(DEVICE)).last_hidden_state
        pooled   = pooling(features)
        logits   = classifier(pooled)
        probs    = torch.softmax(logits, dim=-1).squeeze()
        conf, idx = probs.max(dim=-1)

    prob_dict = {class_names[i]: round(probs[i].item(), 4) for i in range(n_classes)}
    return idx.item(), conf.item(), prob_dict

# ── 7. RUN ON TEST SET ────────────────────────────────────────────────
print("🔄 Running inference on test set...")
with open(TEST_JSON) as f:
    test_data = json.load(f)

records      = []
all_true     = []
all_pred     = []
all_conf     = []

for utt_id, item in test_data.items():
    wav_path = item["wav"]
    true_label = item["emotion"]

    if not os.path.exists(wav_path):
        print(f"   Missing: {wav_path}")
        continue

    try:
        pred_idx, conf, prob_dict = predict_file(wav_path)
        pred_label = class_names[pred_idx]
        correct    = pred_label == true_label

        all_true.append(true_label)
        all_pred.append(pred_label)
        all_conf.append(conf)

        records.append({
            "id":         utt_id,
            "file":       os.path.basename(wav_path),
            "true":       true_label,
            "predicted":  pred_label,
            "correct":    correct,
            "confidence": round(conf, 4),
            **{f"p_{k}": v for k, v in prob_dict.items()},
        })
    except Exception as e:
        print(f"  Error on {wav_path}: {e}")

print(f"Processed {len(records)} samples\n")

# ── 8. SAVE PREDICTIONS CSV ───────────────────────────────────────────
df = pd.DataFrame(records)
csv_path = os.path.join(OUTPUT_DIR, "predictions.csv")
df.to_csv(csv_path, index=False)
print(f"💾 Predictions saved → {csv_path}")

# ── 9. OVERALL ACCURACY ───────────────────────────────────────────────
accuracy = (df["correct"].sum() / len(df)) * 100
print(f"\nOverall Accuracy: {accuracy:.2f}%")
print(f"   Total samples:    {len(df)}")
print(f"   Correct:          {df['correct'].sum()}")
print(f"   Wrong:            {(~df['correct']).sum()}")

# ── 10. PER-CLASS ACCURACY ────────────────────────────────────────────
print("\n📊 Per-Class Accuracy:")
print(f"  {'Emotion':<12} {'Correct':>8} {'Total':>8} {'Accuracy':>10} {'Avg Conf':>10}")
print("  " + "-" * 52)
for emotion in sorted(class_names):
    subset   = df[df["true"] == emotion]
    correct  = subset["correct"].sum()
    total    = len(subset)
    acc      = correct / total * 100 if total > 0 else 0
    avg_conf = subset["confidence"].mean() if total > 0 else 0
    print(f"  {emotion:<12} {correct:>8} {total:>8} {acc:>9.1f}% {avg_conf:>9.3f}")

# ── 11. CLASSIFICATION REPORT ─────────────────────────────────────────
print("\n📋 Classification Report:")
print(classification_report(all_true, all_pred, target_names=sorted(class_names), digits=3))

# ── 12. CONFUSION MATRIX ─────────────────────────────────────────────
print("📊 Generating confusion matrix...")
sorted_labels = sorted(class_names)
cm = confusion_matrix(all_true, all_pred, labels=sorted_labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)   # row-normalize

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=sorted_labels, yticklabels=sorted_labels,
    ax=axes[0], linewidths=0.5, linecolor="gray",
    annot_kws={"size": 11},
)
axes[0].set_title("Confusion Matrix — Raw Counts", fontsize=14, fontweight="bold", pad=12)
axes[0].set_xlabel("Predicted", fontsize=12)
axes[0].set_ylabel("True", fontsize=12)
axes[0].tick_params(axis="x", rotation=45)
axes[0].tick_params(axis="y", rotation=0)

# Normalised (recall per class)
sns.heatmap(
    cm_norm, annot=True, fmt=".2f", cmap="YlOrRd",
    xticklabels=sorted_labels, yticklabels=sorted_labels,
    ax=axes[1], linewidths=0.5, linecolor="gray",
    vmin=0, vmax=1, annot_kws={"size": 11},
)
axes[1].set_title("Confusion Matrix — Normalised (Recall)", fontsize=14, fontweight="bold", pad=12)
axes[1].set_xlabel("Predicted", fontsize=12)
axes[1].set_ylabel("True", fontsize=12)
axes[1].tick_params(axis="x", rotation=45)
axes[1].tick_params(axis="y", rotation=0)

plt.suptitle(f"Speech Emotion Recognition  |  Accuracy: {accuracy:.1f}%",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()

cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Confusion matrix saved → {cm_path}")

# ── 13. CONFIDENCE DISTRIBUTION PLOT ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correct vs wrong confidence
conf_correct = df[df["correct"]]["confidence"]
conf_wrong   = df[~df["correct"]]["confidence"]

axes[0].hist(conf_correct, bins=20, alpha=0.7, color="#2ecc71", label=f"Correct (n={len(conf_correct)})")
axes[0].hist(conf_wrong,   bins=20, alpha=0.7, color="#e74c3c", label=f"Wrong   (n={len(conf_wrong)})")
axes[0].set_xlabel("Confidence", fontsize=12)
axes[0].set_ylabel("Count", fontsize=12)
axes[0].set_title("Confidence Distribution", fontsize=13, fontweight="bold")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Per-class accuracy bar chart
per_class_acc = []
for emotion in sorted_labels:
    subset = df[df["true"] == emotion]
    acc = subset["correct"].mean() * 100 if len(subset) > 0 else 0
    per_class_acc.append(acc)

colors = ["#2ecc71" if a >= 60 else "#e67e22" if a >= 40 else "#e74c3c"
          for a in per_class_acc]
bars = axes[1].bar(sorted_labels, per_class_acc, color=colors, edgecolor="white", linewidth=0.8)
axes[1].axhline(y=accuracy, color="navy", linestyle="--", linewidth=1.5,
                label=f"Overall {accuracy:.1f}%")
axes[1].set_xlabel("Emotion", fontsize=12)
axes[1].set_ylabel("Accuracy (%)", fontsize=12)
axes[1].set_title("Per-Class Accuracy", fontsize=13, fontweight="bold")
axes[1].set_ylim(0, 110)
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)
for bar, acc in zip(bars, per_class_acc):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 1.5,
                 f"{acc:.0f}%", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
dist_path = os.path.join(OUTPUT_DIR, "confidence_and_accuracy.png")
plt.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Distribution plot saved → {dist_path}")

# ── 14. SAMPLE PREDICTIONS PREVIEW ───────────────────────────────────
print("\nSample Predictions (first 20):")
print(df[["file", "true", "predicted", "correct", "confidence"]].head(20).to_string(index=False))

print(f"\n All outputs saved to: {OUTPUT_DIR}")
print(f"   - predictions.csv")
print(f"   - confusion_matrix.png")
print(f"   - confidence_and_accuracy.png")

In [ ]:
%cd /kaggle/working/emotion_classification
!python evaluate.py